# Composite stress tests

Stack **rates**, **credit**, and **FX** using **`compose_scenarios`**, inspect **historical replay** templates, and see how **priority** orders specs when merging.

## Concept

- **Composition**: `compose_scenarios` accepts a JSON **list** of `ScenarioSpec` objects and returns a **single** spec whose operations are merged per engine rules.
- **Priority**: each spec carries an integer **`priority`** (lower runs earlier in composition—see merged ordering in printed JSON).
- **Historical templates**: built-ins such as **`gfc_2008`**, **`covid_2020`**, **`rate_shock_2022`** bundle multiple risk factors; **`list_template_components`** exposes rates/credit/vol/FX building blocks.
- **FX in JSON**: Python `MarketContext.to_json()` may omit ad-hoc `FxMatrix` quotes; for a self-contained FX shock, inject an **`fx`** block with **`quotes: [[base, quote, rate], ...]`** into the parsed market snapshot.


### Composed vs sequential application

**`compose_scenarios`** merges multiple `ScenarioSpec` values into **one** spec. Each input spec has an integer **`priority`**; the merged **`operations`** list reflects the engine’s ordering (**lower priority number ⇒ earlier in the merged stack**—print the numbered operations below to verify).

**Sequential** stress means calling **`apply_scenario_to_market`** on the **base** market, then again on the returned **`market_json`**, and so on—the timeline is whatever order you choose in code. **Composed** stress applies **all** merged operations in **one** call. Because discounting, credit, and FX steps **do not always commute**, the **terminal market** from “compose once” vs “apply three times” can differ; comparing the two is a useful sanity check.

In [ ]:
import json

from datetime import date

from finstack.scenarios import (
    list_builtin_templates,
    list_builtin_template_metadata,
    list_template_components,
    build_from_template,
    build_template_component,
    build_scenario_spec,
    compose_scenarios,
    apply_scenario_to_market,
    validate_scenario_spec,
)
print("apply_scenario_to_market:", callable(apply_scenario_to_market), "build_template_component:", callable(build_template_component))

print("Historical / stress templates:", list_builtin_templates())
meta = list_builtin_template_metadata()
print("Metadata JSON chars:", len(meta))

for tid in ("gfc_2008", "covid_2020", "rate_shock_2022"):
    try:
        if tid in list_builtin_templates():
            comps = list_template_components(tid)
            print(f"Template {tid} components:", comps)
    except Exception as e:
        print("Template inspect failed", tid, e)

rates_json = build_scenario_spec(
    "step_rates",
    json.dumps(
        [
            {
                "kind": "curve_parallel_bp",
                "curve_kind": "discount",
                "curve_id": "USD-OIS",
                "bp": 40.0,
            }
        ]
    ),
    "Rates +40bp",
    "",
    priority=10,
)
credit_json = build_scenario_spec(
    "step_credit",
    json.dumps(
        [
            {
                "kind": "curve_parallel_bp",
                "curve_kind": "par_cds",
                "curve_id": "CORP-HAZARD",
                "discount_curve_id": "USD-OIS",
                "bp": 70.0,
            }
        ]
    ),
    "Credit +70bp",
    "",
    priority=20,
)
fx_json = build_scenario_spec(
    "step_fx",
    json.dumps([{"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": 3.0}]),
    "FX +3% EURUSD",
    "",
    priority=5,
)

specs_list = [json.loads(rates_json), json.loads(credit_json), json.loads(fx_json)]
composed = compose_scenarios(json.dumps(specs_list))
comp_obj = json.loads(composed)
print("Composed scenario id:", comp_obj.get("id"))
print("Composed operations count:", len(comp_obj.get("operations", [])))
print("validate composed:", validate_scenario_spec(composed))
print("First 500 chars of composed:", composed[:500])
print("Composed operations (verify merge / priority ordering):")
for i, op in enumerate(comp_obj.get("operations", [])):
    tag = op.get("curve_id") or op.get("base") or op.get("curve_kind")
    print(f"  [{i}] kind={op.get('kind')!r} ref={tag!r}")


In [ ]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack.scenarios import apply_scenario_to_market, build_scenario_spec, compose_scenarios

AS_OF = "2025-01-15"
bd = date(2025, 1, 15)

usd = DiscountCurve("USD-OIS", bd, [(0.0, 1.0), (1.0, 0.985), (5.0, 0.90)])
hz = HazardCurve("CORP-HAZARD", bd, [(0.0, 0.0), (1.0, 0.02), (5.0, 0.028)], recovery_rate=0.4)
mc = MarketContext().insert(usd).insert(hz)
state = json.loads(mc.to_json())
state["fx"] = {
    "config": {"pivot_currency": "USD", "enable_triangulation": False, "cache_capacity": 256},
    "quotes": [["EUR", "USD", 1.10]],
}
market_json = json.dumps(state)
print("Market with manual FX quotes; EUR/USD=1.10")

# Multi-step composition: explicit priorities (FX first numerically? lower = higher priority — we set FX priority 5, rates 10, credit 20)
rates = build_scenario_spec("r", json.dumps([{"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": 35.0}]), "r", "", 10)
credit = build_scenario_spec("c", json.dumps([{"kind": "curve_parallel_bp", "curve_kind": "par_cds", "curve_id": "CORP-HAZARD", "discount_curve_id": "USD-OIS", "bp": 60.0}]), "c", "", 20)
fx = build_scenario_spec("f", json.dumps([{"kind": "market_fx_pct", "base": "EUR", "quote": "USD", "pct": 4.0}]), "f", "", 5)
composed = compose_scenarios(json.dumps([json.loads(rates), json.loads(credit), json.loads(fx)]))
comp2 = json.loads(composed)
print("Composed spec priorities embedded; merged id:", comp2["id"])
print("Merged operation stack:")
for i, op in enumerate(comp2.get("operations", [])):
    tag = op.get("curve_id") or op.get("base") or op.get("curve_kind")
    print(f"  [{i}] kind={op.get('kind')!r} ref={tag!r}")

res = apply_scenario_to_market(composed, market_json, AS_OF)
print("operations_applied:", res["operations_applied"], "warnings:", res["warnings"])

st = json.loads(res["market_json"])
for c in st["curves"]:
    if c["type"] == "discount" and c["id"] == "USD-OIS":
        print("USD-OIS first DF after:", c["knot_points"][1])
    if c["type"] == "hazard" and c["id"] == "CORP-HAZARD":
        print("CORP-HAZARD knot after (sample):", c["knot_points"][:3])
print("FX quotes after stress:", st.get("fx", {}).get("quotes"))


def usd_ois_df_at_1y(mj: str):
    for c in json.loads(mj).get("curves", []):
        if c.get("type") == "discount" and c.get("id") == "USD-OIS" and c.get("knot_points"):
            return float(c["knot_points"][1][1])
    return None


def eurusd_rate(mj: str):
    quotes = json.loads(mj).get("fx", {}).get("quotes") or []
    return float(quotes[0][2]) if quotes else None


# Sequential application in priority order (5 → 10 → 20) to mirror typical composed ordering
seq_mj = market_json
for label, spec in (("FX +4%", fx), ("Rates +35bp", rates), ("Credit +60bp", credit)):
    step = apply_scenario_to_market(spec, seq_mj, AS_OF)
    seq_mj = step["market_json"]
    print(f"Sequential {label}: operations_applied={step['operations_applied']}")

print(
    "Sanity check — USD-OIS DF@1Y knot | composed vs sequential:",
    usd_ois_df_at_1y(res["market_json"]),
    usd_ois_df_at_1y(seq_mj),
)
print(
    "Sanity check — EUR/USD quote | composed vs sequential:",
    eurusd_rate(res["market_json"]),
    eurusd_rate(seq_mj),
)

# Second pass: apply another small parallel rate shock to already stressed market (sequential application)
follow = build_scenario_spec("f2", json.dumps([{"kind": "curve_parallel_bp", "curve_kind": "discount", "curve_id": "USD-OIS", "bp": 10.0}]), "x", "")
res2 = apply_scenario_to_market(follow, res["market_json"], AS_OF)
print("Second apply operations_applied:", res2["operations_applied"])
d2 = json.loads(res2["market_json"])
for c in d2["curves"]:
    if c["type"] == "discount" and c["id"] == "USD-OIS":
        print("USD-OIS knot after 2nd shock:", c["knot_points"][1])
        break


## Takeaways

- **`compose_scenarios`** merges multiple JSON specs—use it for **rates + credit + FX** (and more) in one application.
- **`priority`** on each spec controls merge ordering; print the composed JSON to confirm the final **`operations`** stack.
- **Historical templates** list via **`list_builtin_templates`**; drill into building blocks with **`list_template_components`**.
- **Chained application** (`apply` on an already stressed `market_json`) models **sequential** stress timelines; composition models **one combined** shock set.
- **FX shocks** need the pair present in **`market_json`**—inject **`fx.quotes`** when the naked `to_json()` snapshot is empty.